<a href="https://colab.research.google.com/github/arunadevnani/AI-Recipe-Assistant-RAG-Gemini/blob/main/GENAIProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install jupyterlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.0 MB/s eta 0:00:00


In [ ]:
from google import genai
from google.colab import userdata


In [ ]:
api_key = userdata.get('GOOGLE_API_KEY')

In [ ]:
client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model ="gemini-2.5-flash",
    contents = "Hi",
    config = {'max_output_tokens':50}
    )
print(response.text)


Hi there! How can I help you today?


In [ ]:
###reading the file

with open('/content/Food_Recipes.txt') as f:
  text = f.read()
  print(text)

Food Recipes Collection

1. Vegetable Fried Rice

Ingredients: - 1 cup cooked rice - 1/2 cup mixed vegetables (carrot,
beans, peas) - 2 tbsp oil - 1 tsp soy sauce - Salt and pepper to taste

Instructions: 1. Heat oil in a pan. 2. Add vegetables and sauté for 3–4
minutes. 3. Add cooked rice and mix well. 4. Add soy sauce, salt, and
pepper. 5. Cook for 2 minutes and serve hot.

2. Tomato Pasta

Ingredients: - 1 cup pasta - 1 cup tomato sauce - 1 tbsp olive oil - 2
cloves garlic - Salt and chili flakes

Instructions: 1. Boil pasta according to package instructions. 2. Heat
olive oil and sauté garlic. 3. Add tomato sauce and cook for 3 minutes.
4. Mix cooked pasta with sauce. 5. Add salt and chili flakes and serve.

3. Fruit Smoothie

Ingredients: - 1 banana - 1/2 cup strawberries - 1 cup milk - 1 tbsp
honey

Instructions: 1. Add all ingredients to a blender. 2. Blend until
smooth. 3. Serve chilled.



In [ ]:
### create chunks of this text
### we should not create embeddings we should not pass all text
chunks = text.split('\n\n')
print(len(chunks))
chunks

10


['Food Recipes Collection',
 '1. Vegetable Fried Rice',
 'Ingredients: - 1 cup cooked rice - 1/2 cup mixed vegetables (carrot,\nbeans, peas) - 2 tbsp oil - 1 tsp soy sauce - Salt and pepper to taste',
 'Instructions: 1. Heat oil in a pan. 2. Add vegetables and sauté for 3–4\nminutes. 3. Add cooked rice and mix well. 4. Add soy sauce, salt, and\npepper. 5. Cook for 2 minutes and serve hot.',
 '2. Tomato Pasta',
 'Ingredients: - 1 cup pasta - 1 cup tomato sauce - 1 tbsp olive oil - 2\ncloves garlic - Salt and chili flakes',
 'Instructions: 1. Boil pasta according to package instructions. 2. Heat\nolive oil and sauté garlic. 3. Add tomato sauce and cook for 3 minutes.\n4. Mix cooked pasta with sauce. 5. Add salt and chili flakes and serve.',
 '3. Fruit Smoothie',
 'Ingredients: - 1 banana - 1/2 cup strawberries - 1 cup milk - 1 tbsp\nhoney',
 'Instructions: 1. Add all ingredients to a blender. 2. Blend until\nsmooth. 3. Serve chilled.\n']

In [ ]:
print(chunks[:2])

['Food Recipes Collection', '1. Vegetable Fried Rice']


In [ ]:
###create embedding model
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import numpy as np
embeddings = model.encode(chunks)
embeddings = np.array(embeddings)
print(embeddings.shape)


(10, 384)


In [ ]:
print(embeddings[0])

[-7.18619004e-02 -7.62825599e-03 -7.82925542e-03  6.11721165e-02
 -8.31918269e-02  5.33146337e-02 -4.02930491e-02 -5.59707433e-02
  2.71267816e-03 -4.25127558e-02  8.13846514e-02 -6.97456449e-02
 -3.93501706e-02 -1.17887892e-01 -2.45052688e-02 -2.90540289e-02
  1.17237344e-01  2.45474204e-02  2.80254856e-02 -1.33046344e-01
  2.20534299e-03  4.42130119e-02  7.67446086e-02  2.14251829e-03
  3.66591848e-02 -1.46351159e-02  4.76501659e-02 -1.27136372e-02
 -9.15364828e-03 -6.36431724e-02 -1.69247966e-02  7.15333677e-04
  4.13298495e-02  2.17891168e-02 -4.68517741e-04 -1.35809155e-02
  9.71507579e-02 -4.47032712e-02  4.32101674e-02 -1.43036188e-03
 -9.28559806e-03  2.29008719e-02  3.60718719e-03  1.17732882e-02
  3.17282113e-03  3.93454060e-02 -7.28411973e-02 -4.77718236e-03
  2.58207582e-02  1.48955211e-02 -9.23714116e-02  1.71194132e-02
 -2.91644186e-02 -5.70401847e-02  7.62622952e-02 -1.57000311e-02
 -5.67363203e-02 -1.29939457e-02 -2.74140202e-02  5.09483106e-02
  2.69658826e-02 -5.44210

In [ ]:
###create faiss - to store embeddings in Vector DataBase
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.2 MB/s eta 0:00:00


In [ ]:
import faiss

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("Vector Stored-", index.ntotal)

Vector Stored- 10


In [ ]:
## Text data- Read - Create Chunks - created embeddings - stored embeddings in Vector DB
###Retrieval function (RAG)
### most relevant recipe asked by user
def retrieve_context(query, top_k = 3):
  query_embeddings = model.encode([query])
  distances,indices = index.search(np.array(query_embeddings),top_k)
  results = [chunks[i] for i in indices[0]]
  return "\n".join(results)


In [ ]:
query = "How do i make Vegetable Fried Rice"
retrieve_context(query,top_k = 3)

'1. Vegetable Fried Rice\nInstructions: 1. Heat oil in a pan. 2. Add vegetables and sauté for 3–4\nminutes. 3. Add cooked rice and mix well. 4. Add soy sauce, salt, and\npepper. 5. Cook for 2 minutes and serve hot.\nIngredients: - 1 cup cooked rice - 1/2 cup mixed vegetables (carrot,\nbeans, peas) - 2 tbsp oil - 1 tsp soy sauce - Salt and pepper to taste'

In [ ]:
def generate_answer(query):
  context = retrieve_context(query, top_k = 3 )

  system_prompt = "You are a expert chef, please answer the query in simple manner."

  user_prompt = f"""
  Answer the following user query - {query} based on the context
  shared {context} to get the recipes of the food for prepartion.
  Use bullet points to answer the query.
  """
  response = client.models.generate_content(
    model ="gemini-2.5-flash",
    contents = f"{system_prompt}\n\n{user_prompt}"
      )
    ##print(response.text)
  return response.text



In [ ]:
while True:
    question = input("\nEnter your recipe questions (type exit to stop): ")

    if question.lower() == "exit":
        print("Thank you for using the BOT")
        break

    answer = generate_answer(question)
    print("\nBot:", answer)


Bot: Alright, aspiring chef! Tomato pasta is a simple, delicious classic, and I can walk you through it easily. Here's how you'll whip up that tasty dish:

**Tomato Pasta**

Here's what you'll need:

*   1 cup pasta
*   1 cup tomato sauce
*   1 tablespoon olive oil
*   2 cloves garlic
*   Salt and chili flakes (to your taste)

And here's how to make it:

*   First, get your pasta boiling! Cook it according to the instructions on its package.
*   While the pasta cooks, heat up your olive oil in a pan and gently sauté the garlic until it smells fragrant.
*   Pour in your tomato sauce and let it cook for about 3 minutes.
*   Once your pasta is ready and drained, mix it right into the sauce.
*   Finally, season with salt and a sprinkle of chili flakes to your liking, then serve it up hot!

Enjoy your homemade tomato pasta!

Bot: I'm sorry, but the provided "Food Recipes Collection" context does not contain a recipe for an omelette. It only includes recipes for pasta and rice. Therefore, I

In [ ]:
###Food_Recipes.txt....>Chunking....>Embeddings........>Faiss Vector DB to store data.......>User question.....>Retrieve relevant recipes(context).....>Context+
query......>By Gemini LLM..........>Final Answer

In [ ]:
import pandas as pd
df = {'country': ['USA','China','India','Japan','Mexico'],
      'population':[140,1400,1000,300,250],
      'GDP':[3.73,25.42,8.9,7.9,29],
      'Revenue':[2300,5400,7800,2300,4500]}
df = pd.DataFrame(df)
print(df)


In [ ]:
table_text = df.to_markdown(index=True)
table_text

In [ ]:
### pass the data to LLM............>LLm will give answer

In [ ]:
def answer(query)
  response = client.models.generate_content(models ="gemini-2.5-flash", contents=f"{syatem_prompt}\n\n{user_prompt}")
  return response.text

question = input("\nEnter your question (type exit to stop): ")
if question.lower() == "exit":
    print("Thank you for using the BOT")

    system_prompt = "You are a Data Analyst to answer user query"

    user_prompt = f"""
    Use the following table {table_text} to answer the user query-
    {question}
    """
    answer =answer(question)
    print("\nBot:", answer)

